# Priority 1 — Qwen Reconstructed Full-Context Extraction

This notebook performs the frozen comparison condition for
`Qwen/Qwen2.5-1.5B-Instruct`.

It does **not** regenerate responses. It reconstructs the original serialized
system/user prompt, appends the recorded completion token IDs, and runs a
teacher-forced forward pass.

The extraction design is frozen to match the existing completion-only study:

- 30 completion tokens immediately before the pivot are the query positions;
- the 250 causally preceding tokens are the retained key window;
- H1–H4 are computed later with the already frozen formulas;
- all labels and exclusions are unchanged.

The output is called **reconstructed full-context replay**, not literal
generation-time activation capture, because original generated token IDs and
the exact Hugging Face revision were not archived.

In [1]:
!pip install -q "transformer_lens>=3.1,<4" transformers huggingface_hub

import gc
import hashlib
import json
import os
import platform
import shutil
import time
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer

assert torch.cuda.is_available(), "Enable a Kaggle GPU before running."

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", version("transformers"))
print("TransformerLens:", version("transformer-lens"))

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 95.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.7 MB/s eta 0:00:00:00:01
GPU: Tesla T4
VRAM GB: 15.64
Python: 3.12.13
PyTorch: 2.10.0+cu128
Transformers: 5.16.1
TransformerLens: 3.8.0


## 1. Locate the final Phase-2 records

In [2]:
def find_unique_file(filenames, preferred_paths=()):
    for preferred in preferred_paths:
        p = Path(preferred)
        if p.exists():
            print("Using preferred input:", p)
            return p

    candidates = []
    for filename in filenames:
        for root in (Path("/kaggle/input"), Path("/kaggle/working")):
            if root.exists():
                candidates.extend(root.rglob(filename))

    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            "None of the requested files were found: "
            + ", ".join(filenames)
        )

    if len(candidates) > 1:
        print("Candidate files:")
        for i, p in enumerate(candidates):
            print(f"  [{i}] {p}")
        raise RuntimeError(
            "Multiple candidate label files were found. "
            "Set PHASE2_PATH manually to the intended final file."
        )

    print("Resolved input:", candidates[0])
    return candidates[0]


def load_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

PHASE2_PATH = find_unique_file(
    filenames=[
        "phase2_labeled_combined_clean.jsonl",
        "phase2_labeled_combined.jsonl",
    ],
    preferred_paths=[
        "/kaggle/input/notebooks/gunjalgupta/phase-2/pivot_research/phase2_labeled_combined.jsonl",
        "/kaggle/working/phase2_labeled_combined_clean.jsonl",
    ],
)

OUTPUT_DIR = Path("/kaggle/working/full_context_qwen")
RECORD_DIR = OUTPUT_DIR / "records"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RECORD_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = OUTPUT_DIR / "full_context_qwen_metadata.jsonl"
ALIGNMENT_PATH = OUTPUT_DIR / "full_context_qwen_alignment_audit.csv"
MANIFEST_PATH = OUTPUT_DIR / "full_context_qwen_manifest.json"
ERRORS_PATH = OUTPUT_DIR / "full_context_qwen_errors.jsonl"

all_records = load_jsonl(PHASE2_PATH)
usable_records = [
    r for r in all_records
    if r.get("label") in {"correction", "hallucination_pivot"}
    and int(r.get("pivot_token_start", -1)) >= 30
]

counts = Counter(r["label"] for r in usable_records)
print("All records:", len(all_records))
print("Usable:", len(usable_records))
print("Labels:", dict(counts))

assert len(all_records) == 155
assert len(usable_records) == 127
assert counts["correction"] == 22
assert counts["hallucination_pivot"] == 105

Using preferred input: /kaggle/input/notebooks/gunjalgupta/phase-2/pivot_research/phase2_labeled_combined.jsonl
All records: 155
Usable: 127
Labels: {'hallucination_pivot': 105, 'correction': 22}


## 2. Load the same model family and extraction stack

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_DTYPE = torch.float16

gc.collect()
torch.cuda.empty_cache()

model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    device="cuda",
    dtype=MODEL_DTYPE,
)
model.eval()
tokenizer = model.tokenizer

N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads
D_HEAD = model.cfg.d_head
D_MODEL = model.cfg.d_model

print("Layers:", N_LAYERS)
print("Heads:", N_HEADS)
print("d_head:", D_HEAD)
print("d_model:", D_MODEL)
print("Tokenizer:", type(tokenizer).__name__)

/tmp/ipykernel_58/3938196761.py:7: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-1.5B-Instruct into HookedTransformer
Layers: 28
Heads: 12
d_head: 128
d_model: 1536
Tokenizer: Qwen2Tokenizer


## 3. Freeze exact prompt reconstruction

In [4]:
SYSTEM_PROMPT = "You are a careful, honest reasoner. When solving a problem:\n1. Start reasoning immediately — commit to a first approach even if uncertain\n2. Work through the steps out loud\n3. Pause mid-way and ask: 'Wait, is this actually right?'\n4. If you catch ANY error or realize something was off, say so explicitly using phrases like:\n   'Wait,' 'Actually,' 'Hmm, let me reconsider,' 'I made an error,' 'Oh wait'\n5. Correct yourself and give the final answer\n\nNever skip steps. Never suppress doubt. Thinking out loud is the goal."
USER_SUFFIX = (
    "\n\nRemember: commit to a first answer, then double-check it. "
    "Show your doubt explicitly."
)

def build_serialized_prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question + USER_SUFFIX},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def encode_generation_prompt(question):
    # This mirrors the original generation call:
    # tokenizer(prompts, truncation=True, max_length=512)
    serialized = build_serialized_prompt(question)

    raw_ids = tokenizer(
        serialized,
        add_special_tokens=True,
        truncation=False,
        return_tensors="pt",
    )["input_ids"][0]

    used_ids = tokenizer(
        serialized,
        add_special_tokens=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )["input_ids"][0]

    return {
        "serialized": serialized,
        "raw_ids": raw_ids,
        "used_ids": used_ids,
        "was_truncated": len(raw_ids) != len(used_ids),
    }


print(build_serialized_prompt("Test question")[:600])
print("System SHA256:", sha256_text(SYSTEM_PROMPT))
print("Chat template SHA256:", sha256_text(tokenizer.chat_template or ""))

<|im_start|>system
You are a careful, honest reasoner. When solving a problem:
1. Start reasoning immediately — commit to a first approach even if uncertain
2. Work through the steps out loud
3. Pause mid-way and ask: 'Wait, is this actually right?'
4. If you catch ANY error or realize something was off, say so explicitly using phrases like:
   'Wait,' 'Actually,' 'Hmm, let me reconsider,' 'I made an error,' 'Oh wait'
5. Correct yourself and give the final answer

Never skip steps. Never suppress doubt. Thinking out loud is the goal.<|im_end|>
<|im_start|>user
Test question

Remember: commit t
System SHA256: 9e2c38df8555d77e4ef01c80a08fbb15adc639ac792d7e8edf37fd3b73588d70
Chat template SHA256: cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f


## 4. Alignment audit

The recorded completion is tokenized exactly as in the existing Phase-2/3
pipeline (`add_special_tokens=False`). The reconstructed sequence is created
by concatenating prompt token IDs with these completion token IDs. This
preserves the existing completion-relative pivot indices and avoids
re-tokenizing across the assistant-prefix boundary.

In [5]:
QUERY_WINDOW_SIZE = 30
MAX_KEY_CONTEXT = 250

alignment_rows = []
prepared = []

for source_idx, r in enumerate(usable_records):
    prompt = encode_generation_prompt(r["prompt_text"])
    prompt_ids = prompt["used_ids"].to(dtype=torch.long)

    completion_ids = tokenizer.encode(
        r["generated_text"],
        add_special_tokens=False,
        return_tensors="pt",
    )[0].to(dtype=torch.long)

    pivot_completion_idx = int(r["pivot_token_start"])
    pivot_full_idx = len(prompt_ids) + pivot_completion_idx
    full_ids = torch.cat([prompt_ids, completion_ids], dim=0)

    query_completion = completion_ids[
        pivot_completion_idx - QUERY_WINDOW_SIZE:
        pivot_completion_idx
    ]
    query_full = full_ids[
        pivot_full_idx - QUERY_WINDOW_SIZE:
        pivot_full_idx
    ]

    query_exact = bool(torch.equal(query_completion, query_full))
    pivot_valid = (
        pivot_completion_idx >= QUERY_WINDOW_SIZE
        and pivot_completion_idx < len(completion_ids)
    )

    neighborhood = tokenizer.decode(
        completion_ids[
            max(0, pivot_completion_idx - 6):
            min(len(completion_ids), pivot_completion_idx + 10)
        ]
    )

    key_start = max(0, pivot_full_idx - MAX_KEY_CONTEXT)
    key_positions = np.arange(key_start, pivot_full_idx)
    prompt_keys = int(np.sum(key_positions < len(prompt_ids)))
    completion_keys = int(len(key_positions) - prompt_keys)

    total_tokens_recorded = r.get("total_tokens")
    total_tokens_match = (
        total_tokens_recorded is None
        or int(total_tokens_recorded) == len(completion_ids)
    )

    key = f'{r["prompt_id"]}#{r.get("sample_idx")}'
    row = {
        "source_idx": source_idx,
        "key": key,
        "prompt_id": r["prompt_id"],
        "sample_idx": r.get("sample_idx"),
        "label": r["label"],
        "prompt_tokens_raw": int(len(prompt["raw_ids"])),
        "prompt_tokens_used": int(len(prompt_ids)),
        "prompt_was_truncated": bool(prompt["was_truncated"]),
        "completion_tokens_reencoded": int(len(completion_ids)),
        "completion_tokens_recorded": (
            None if total_tokens_recorded is None
            else int(total_tokens_recorded)
        ),
        "completion_length_matches_record": bool(total_tokens_match),
        "pivot_completion_idx": pivot_completion_idx,
        "pivot_full_idx": int(pivot_full_idx),
        "pivot_valid": bool(pivot_valid),
        "query_token_ids_exact": bool(query_exact),
        "prompt_keys_in_250_window": prompt_keys,
        "completion_keys_in_250_window": completion_keys,
        "pivot_neighborhood": neighborhood,
    }
    alignment_rows.append(row)

    prepared.append({
        "source_idx": source_idx,
        "record": r,
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "full_ids": full_ids,
        "pivot_full_idx": pivot_full_idx,
        "prompt_info": prompt,
        "alignment": row,
    })

alignment_df = pd.DataFrame(alignment_rows)
alignment_df.to_csv(ALIGNMENT_PATH, index=False)

print(alignment_df[
    [
        "prompt_was_truncated",
        "completion_length_matches_record",
        "pivot_valid",
        "query_token_ids_exact",
    ]
].apply(pd.Series.value_counts).fillna(0))

assert alignment_df["pivot_valid"].all()
assert alignment_df["query_token_ids_exact"].all()

failure_rate = 1.0 - alignment_df[
    "completion_length_matches_record"
].mean()
print("Recorded completion-length mismatch rate:", f"{failure_rate:.1%}")

if failure_rate > 0.05:
    raise RuntimeError(
        "More than 5% of completions fail the archived token-length check. "
        "Stop and investigate before extraction."
    )

print("Alignment audit saved:", ALIGNMENT_PATH)

       prompt_was_truncated  completion_length_matches_record  pivot_valid  \
False                 127.0                               0.0          0.0   
True                    0.0                             127.0        127.0   

       query_token_ids_exact  
False                    0.0  
True                   127.0  
Recorded completion-length mismatch rate: 0.0%
Alignment audit saved: /kaggle/working/full_context_qwen/full_context_qwen_alignment_audit.csv


## 5. Low-memory full-context extraction

In [6]:
def extract_full_context(prepared_record):
    full_ids = prepared_record["full_ids"].unsqueeze(0).to("cuda")
    pivot_idx = int(prepared_record["pivot_full_idx"])

    query_start = pivot_idx - QUERY_WINDOW_SIZE
    query_end = pivot_idx
    key_real_start = max(0, pivot_idx - MAX_KEY_CONTEXT)
    key_end = pivot_idx

    n_real_keys = key_end - key_real_start
    valid_key_start = MAX_KEY_CONTEXT - n_real_keys

    weights_by_layer = [None] * N_LAYERS
    outputs_by_layer = [None] * N_LAYERS

    def make_pattern_hook(layer_idx):
        def hook_fn(pattern, hook):
            sliced = pattern[
                0,
                :,
                query_start:query_end,
                key_real_start:key_end,
            ]
            sliced_cpu = (
                sliced.detach()
                .to("cpu", dtype=torch.float32)
                .numpy()
            )
            padded = np.zeros(
                (N_HEADS, QUERY_WINDOW_SIZE, MAX_KEY_CONTEXT),
                dtype=np.float32,
            )
            padded[:, :, valid_key_start:] = sliced_cpu
            weights_by_layer[layer_idx] = padded
            return None
        return hook_fn

    def make_z_hook(layer_idx):
        def hook_fn(z, hook):
            sliced = z[0, query_start:query_end, :, :]
            outputs_by_layer[layer_idx] = (
                sliced.detach()
                .to("cpu", dtype=torch.float32)
                .numpy()
            )
            return None
        return hook_fn

    hooks = []
    for layer in range(N_LAYERS):
        hooks.append((
            f"blocks.{layer}.attn.hook_pattern",
            make_pattern_hook(layer),
        ))
        hooks.append((
            f"blocks.{layer}.attn.hook_z",
            make_z_hook(layer),
        ))

    with torch.inference_mode():
        model.run_with_hooks(
            full_ids,
            fwd_hooks=hooks,
            return_type=None,
        )

    missing_weights = [
        i for i, value in enumerate(weights_by_layer)
        if value is None
    ]
    missing_outputs = [
        i for i, value in enumerate(outputs_by_layer)
        if value is None
    ]
    if missing_weights or missing_outputs:
        raise RuntimeError(
            f"Missing hooks: weights={missing_weights}, "
            f"outputs={missing_outputs}"
        )

    weights = np.stack(weights_by_layer, axis=0)
    outputs = np.stack(outputs_by_layer, axis=0)

    del full_ids, weights_by_layer, outputs_by_layer, hooks
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "weights": weights,
        "outputs": outputs,
        "valid_key_start": valid_key_start,
        "n_real_keys": n_real_keys,
        "full_seq_len": int(len(prepared_record["full_ids"])),
    }

## 6. Mandatory one-record dry run

In [7]:
test_item = prepared[0]
print(
    test_item["record"]["prompt_id"],
    test_item["record"].get("sample_idx"),
    test_item["record"]["label"],
)

start = time.time()
test_result = extract_full_context(test_item)
elapsed = time.time() - start

assert test_result["weights"].shape == (
    N_LAYERS,
    N_HEADS,
    QUERY_WINDOW_SIZE,
    MAX_KEY_CONTEXT,
)
assert test_result["outputs"].shape == (
    N_LAYERS,
    QUERY_WINDOW_SIZE,
    N_HEADS,
    D_HEAD,
)

print("Elapsed seconds:", round(elapsed, 2))
print("Weights:", test_result["weights"].shape)
print("Outputs:", test_result["outputs"].shape)
print("Full sequence length:", test_result["full_seq_len"])
print("Real keys:", test_result["n_real_keys"])

del test_result
gc.collect()
torch.cuda.empty_cache()

mis_03 8 hallucination_pivot
Elapsed seconds: 1.03
Weights: (28, 12, 30, 250)
Outputs: (28, 30, 12, 128)
Full sequence length: 567
Real keys: 250


## 7. Extract all 127 records

In [8]:
RESET_OUTPUTS = True

if RESET_OUTPUTS and RECORD_DIR.exists():
    shutil.rmtree(RECORD_DIR)
    RECORD_DIR.mkdir(parents=True, exist_ok=True)

metadata = []
errors = []
start_time = time.time()

for item in tqdm(prepared, desc="Qwen full-context extraction"):
    r = item["record"]
    try:
        result = extract_full_context(item)
        out_idx = len(metadata)
        record_file = RECORD_DIR / f"record_{out_idx:03d}.npz"

        np.savez_compressed(
            record_file,
            weights=result["weights"],
            outputs=result["outputs"],
        )

        alignment = item["alignment"]
        metadata.append({
            "idx": out_idx,
            "source_idx": item["source_idx"],
            "record_file": record_file.name,
            "prompt_id": r["prompt_id"],
            "sample_idx": r.get("sample_idx"),
            "category": r.get("category"),
            "label": r["label"],
            "label_method": r.get("label_method"),
            "human_verified": r.get("human_verified", False),
            "pivot_type": r.get("pivot_type"),
            "pivot_phrase": r.get("pivot_phrase"),
            "pivot_completion_idx": int(r["pivot_token_start"]),
            "pivot_full_idx": int(item["pivot_full_idx"]),
            "prompt_token_count": int(len(item["prompt_ids"])),
            "completion_token_count": int(len(item["completion_ids"])),
            "full_seq_len": int(result["full_seq_len"]),
            "query_window_size": QUERY_WINDOW_SIZE,
            "max_key_context": MAX_KEY_CONTEXT,
            "valid_key_start": int(result["valid_key_start"]),
            "n_real_keys": int(result["n_real_keys"]),
            "prompt_keys_in_window": int(
                alignment["prompt_keys_in_250_window"]
            ),
            "completion_keys_in_window": int(
                alignment["completion_keys_in_250_window"]
            ),
            "prompt_was_truncated": bool(
                alignment["prompt_was_truncated"]
            ),
            "completion_length_matches_record": bool(
                alignment["completion_length_matches_record"]
            ),
            "model_name": MODEL_NAME,
            "condition": "reconstructed_full_context",
        })

        del result
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as exc:
        errors.append({
            "source_idx": item["source_idx"],
            "prompt_id": r.get("prompt_id"),
            "sample_idx": r.get("sample_idx"),
            "label": r.get("label"),
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        print(
            "ERROR",
            r.get("prompt_id"),
            r.get("sample_idx"),
            type(exc).__name__,
            str(exc),
        )

elapsed = time.time() - start_time

with METADATA_PATH.open("w", encoding="utf-8") as f:
    for row in metadata:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with ERRORS_PATH.open("w", encoding="utf-8") as f:
    for row in errors:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Successful:", len(metadata))
print("Errors:", len(errors))
print("Elapsed minutes:", round(elapsed / 60, 2))

assert len(metadata) == 127
assert not errors

Qwen full-context extraction:   0%|          | 0/127 [00:00<?, ?it/s]

Successful: 127
Errors: 0
Elapsed minutes: 6.62


## 8. Save manifest and validate outputs

In [9]:
model_commit = getattr(model.cfg, "original_architecture", None)
tokenizer_commit = getattr(tokenizer, "init_kwargs", {}).get(
    "_commit_hash"
)

manifest = {
    "study_condition": "reconstructed_full_context",
    "model_name": MODEL_NAME,
    "model_dtype": str(MODEL_DTYPE),
    "model_revision_pinned_in_original_study": False,
    "tokenizer_commit_observed_now": tokenizer_commit,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": version("transformers"),
    "transformer_lens_version": version("transformer-lens"),
    "tokenizer_class": type(tokenizer).__name__,
    "system_prompt_sha256": sha256_text(SYSTEM_PROMPT),
    "chat_template_sha256": sha256_text(
        tokenizer.chat_template or ""
    ),
    "phase2_path": str(PHASE2_PATH),
    "n_phase2_records": len(all_records),
    "n_requested": len(usable_records),
    "n_extracted": len(metadata),
    "n_errors": len(errors),
    "query_window_size": QUERY_WINDOW_SIZE,
    "max_key_context": MAX_KEY_CONTEXT,
    "label_counts": dict(Counter(m["label"] for m in metadata)),
    "prompt_truncation_count": int(
        sum(m["prompt_was_truncated"] for m in metadata)
    ),
    "completion_length_mismatch_count": int(
        sum(
            not m["completion_length_matches_record"]
            for m in metadata
        )
    ),
    "elapsed_minutes": elapsed / 60,
    "reconstruction_note": (
        "Prompt token IDs were reconstructed from the archived system/user "
        "text and current checkpoint chat template. Completion token IDs "
        "were re-encoded from archived generated_text because original "
        "generation token IDs were not saved."
    ),
}

with MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

for m in tqdm(metadata, desc="Validating saved tensors"):
    with np.load(RECORD_DIR / m["record_file"]) as archive:
        assert archive["weights"].shape == (
            N_LAYERS,
            N_HEADS,
            QUERY_WINDOW_SIZE,
            MAX_KEY_CONTEXT,
        )
        assert archive["outputs"].shape == (
            N_LAYERS,
            QUERY_WINDOW_SIZE,
            N_HEADS,
            D_HEAD,
        )

print("READY FOR PAIRED COMPARISON")
print("Records:", RECORD_DIR)
print("Metadata:", METADATA_PATH)
print("Alignment audit:", ALIGNMENT_PATH)
print("Manifest:", MANIFEST_PATH)

Validating saved tensors:   0%|          | 0/127 [00:00<?, ?it/s]

READY FOR PAIRED COMPARISON
Records: /kaggle/working/full_context_qwen/records
Metadata: /kaggle/working/full_context_qwen/full_context_qwen_metadata.jsonl
Alignment audit: /kaggle/working/full_context_qwen/full_context_qwen_alignment_audit.csv
Manifest: /kaggle/working/full_context_qwen/full_context_qwen_manifest.json


In [10]:
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED
from IPython.display import FileLink, display

source_dir = Path("/kaggle/working/full_context_qwen")
zip_path = Path("/kaggle/working/full_context_qwen_complete.zip")

# Confirm the extraction folder exists.
if not source_dir.exists():
    raise FileNotFoundError(f"Folder does not exist: {source_dir}")

files = sorted(p for p in source_dir.rglob("*") if p.is_file())
npz_files = [p for p in files if p.suffix == ".npz"]

print(f"Source folder: {source_dir}")
print(f"Total files found: {len(files)}")
print(f"NPZ files found: {len(npz_files)}")

if len(npz_files) != 127:
    raise RuntimeError(
        f"Expected 127 NPZ record files, but found {len(npz_files)}."
    )

# Remove an incomplete ZIP left by a previous attempt.
if zip_path.exists():
    zip_path.unlink()

print("\nCreating ZIP. This may take a few minutes...")

with ZipFile(zip_path, mode="w", compression=ZIP_STORED) as archive:
    for index, file_path in enumerate(files, start=1):
        # The ZIP will contain full_context_qwen/... rather than loose files.
        archive_name = file_path.relative_to(source_dir.parent)
        archive.write(file_path, arcname=archive_name)

        if index % 25 == 0 or index == len(files):
            print(f"Archived {index}/{len(files)} files")

# Verify integrity.
with ZipFile(zip_path, "r") as archive:
    corrupted_member = archive.testzip()
    archived_npz = [
        name for name in archive.namelist()
        if name.endswith(".npz")
    ]

if corrupted_member is not None:
    raise RuntimeError(f"Corrupted ZIP entry: {corrupted_member}")

if len(archived_npz) != 127:
    raise RuntimeError(
        f"ZIP contains {len(archived_npz)} NPZ files instead of 127."
    )

print("\nZIP successfully created.")
print(f"Path: {zip_path}")
print(f"Size: {zip_path.stat().st_size / (1024**2):.2f} MiB")
print(f"NPZ files inside ZIP: {len(archived_npz)}")
print("Integrity check: PASS")

display(FileLink(str(zip_path)))

Source folder: /kaggle/working/full_context_qwen
Total files found: 131
NPZ files found: 127

Creating ZIP. This may take a few minutes...
Archived 25/131 files
Archived 50/131 files
Archived 75/131 files
Archived 100/131 files
Archived 125/131 files
Archived 131/131 files

ZIP successfully created.
Path: /kaggle/working/full_context_qwen_complete.zip
Size: 985.51 MiB
NPZ files inside ZIP: 127
Integrity check: PASS


/kaggle/working/full_context_qwen_complete.zip

In [11]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

BASE_DIR = Path("/kaggle/working/full_context_qwen")
RECORDS_DIR = BASE_DIR / "records"

# Find metadata even if its exact location differs slightly.
metadata_candidates = [
    BASE_DIR / "full_context_qwen_metadata.jsonl",
    Path("/kaggle/working/full_context_qwen_metadata.jsonl"),
]

METADATA_PATH = next(
    (path for path in metadata_candidates if path.exists()),
    None,
)

if METADATA_PATH is None:
    raise FileNotFoundError(
        "Could not locate full_context_qwen_metadata.jsonl"
    )

OUTPUT_DIR = Path("/kaggle/working/qwen_compact_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_NPZ = OUTPUT_DIR / "qwen_full_context_features.npz"
FEATURE_CSV = OUTPUT_DIR / "qwen_full_context_features.csv"
PROMPT_FEATURE_NPZ = OUTPUT_DIR / "qwen_prompt_attention_features.npz"
SUMMARY_JSON = OUTPUT_DIR / "qwen_compact_feature_manifest.json"
OUTPUT_ZIP = Path("/kaggle/working/qwen_compact_analysis_bundle.zip")


# ============================================================
# Load metadata
# ============================================================

metadata = []

with METADATA_PATH.open("r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line:
            metadata.append(json.loads(line))

metadata = sorted(metadata, key=lambda row: int(row["idx"]))

print(f"Metadata rows: {len(metadata)}")

if len(metadata) != 127:
    raise RuntimeError(
        f"Expected 127 Qwen records, found {len(metadata)}"
    )


# ============================================================
# Feature extraction
# ============================================================

def normalized_structural_entropy(
    head_attention: np.ndarray,
    valid_key_start: int,
    n_real_keys: int,
) -> float:
    """
    Compute final H3 for one head.

    head_attention shape:
        query_positions x key_positions

    For query q, the structurally available causal-key count is:

        K_q = n_real_keys - (Q - 1 - q)

    Attention is restricted to those available captured keys,
    renormalized, and entropy is divided by log(K_q).
    """
    query_count = head_attention.shape[0]
    entropies = []

    for q in range(query_count):
        k_q = n_real_keys - (query_count - 1 - q)

        if k_q <= 1:
            entropies.append(0.0)
            continue

        key_end = valid_key_start + k_q

        probabilities = np.asarray(
            head_attention[q, valid_key_start:key_end],
            dtype=np.float64,
        )

        probabilities = np.clip(probabilities, 0.0, None)
        mass = probabilities.sum()

        if mass <= 0:
            entropies.append(0.0)
            continue

        probabilities /= mass
        positive = probabilities > 0

        entropy = -np.sum(
            probabilities[positive]
            * np.log(probabilities[positive])
        )

        normalized_entropy = entropy / np.log(k_q)
        entropies.append(float(normalized_entropy))

    return float(np.mean(entropies))


all_features = []
all_prompt_attention = []
output_rows = []

feature_names = None
prompt_feature_names = None

for position, row in enumerate(metadata):
    record_path = RECORDS_DIR / row["record_file"]

    if not record_path.exists():
        raise FileNotFoundError(f"Missing tensor file: {record_path}")

    with np.load(record_path) as archive:
        weights = archive["weights"].astype(np.float32, copy=False)
        outputs = archive["outputs"].astype(np.float32, copy=False)

    # Expected Qwen shapes:
    # weights: layers x heads x queries x keys
    # outputs: layers x queries x heads x d_head
    if weights.ndim != 4 or outputs.ndim != 4:
        raise RuntimeError(
            f"Unexpected tensor dimensions in {record_path.name}: "
            f"weights={weights.shape}, outputs={outputs.shape}"
        )

    n_layers, n_heads, query_count, key_count = weights.shape

    if outputs.shape[:3] != (n_layers, query_count, n_heads):
        raise RuntimeError(
            f"Weight/output shape mismatch in {record_path.name}"
        )

    valid_key_start = int(row["valid_key_start"])
    n_real_keys = int(row["n_real_keys"])
    prompt_keys = int(row.get("prompt_keys_in_window", 0))

    # --------------------------------------------------------
    # H1: earliest 50 retained real keys
    # --------------------------------------------------------
    h1_end = valid_key_start + min(50, n_real_keys)

    h1 = weights[
        :,
        :,
        :,
        valid_key_start:h1_end,
    ].sum(axis=-1).mean(axis=-1)

    # Shape: layers x heads

    # --------------------------------------------------------
    # H2: mean output-vector L2 magnitude
    # --------------------------------------------------------
    h2 = np.linalg.norm(outputs, axis=-1).mean(axis=1)

    # Shape: layers x heads

    # --------------------------------------------------------
    # H3: structurally normalized attention entropy
    # --------------------------------------------------------
    h3 = np.zeros((n_layers, n_heads), dtype=np.float32)

    for layer in range(n_layers):
        for head in range(n_heads):
            h3[layer, head] = normalized_structural_entropy(
                weights[layer, head],
                valid_key_start=valid_key_start,
                n_real_keys=n_real_keys,
            )

    # --------------------------------------------------------
    # H4: late-layer / early-layer H2 ratio
    # Qwen: early layers 0–8, late layers 18–27
    # --------------------------------------------------------
    early_magnitude = float(h2[0:9].mean())
    late_magnitude = float(h2[18:28].mean())

    h4 = (
        late_magnitude / early_magnitude
        if early_magnitude > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Exploratory prompt-token attention
    # --------------------------------------------------------
    prompt_attention = np.zeros(
        (n_layers, n_heads),
        dtype=np.float32,
    )

    if prompt_keys > 0:
        prompt_end = valid_key_start + prompt_keys

        prompt_attention = weights[
            :,
            :,
            :,
            valid_key_start:prompt_end,
        ].sum(axis=-1).mean(axis=-1)

    # --------------------------------------------------------
    # Final frozen feature vector: 336 + 336 + 336 + 1
    # --------------------------------------------------------
    feature_vector = np.concatenate(
        [
            h1.reshape(-1),
            h2.reshape(-1),
            h3.reshape(-1),
            np.asarray([h4], dtype=np.float32),
        ]
    ).astype(np.float32)

    expected_feature_count = n_layers * n_heads * 3 + 1

    if feature_vector.shape[0] != expected_feature_count:
        raise RuntimeError(
            f"Unexpected feature count: {feature_vector.shape[0]}"
        )

    if not np.all(np.isfinite(feature_vector)):
        raise RuntimeError(
            f"Non-finite feature found in {record_path.name}"
        )

    if feature_names is None:
        feature_names = (
            [
                f"H1_L{layer:02d}_H{head:02d}"
                for layer in range(n_layers)
                for head in range(n_heads)
            ]
            + [
                f"H2_L{layer:02d}_H{head:02d}"
                for layer in range(n_layers)
                for head in range(n_heads)
            ]
            + [
                f"H3_L{layer:02d}_H{head:02d}"
                for layer in range(n_layers)
                for head in range(n_heads)
            ]
            + ["H4_late_early_ratio"]
        )

        prompt_feature_names = [
            f"PROMPT_ATTN_L{layer:02d}_H{head:02d}"
            for layer in range(n_layers)
            for head in range(n_heads)
        ]

    all_features.append(feature_vector)
    all_prompt_attention.append(prompt_attention.reshape(-1))

    output_row = {
        "idx": int(row["idx"]),
        "source_idx": int(row["source_idx"]),
        "prompt_id": row["prompt_id"],
        "sample_idx": row.get("sample_idx"),
        "category": row.get("category"),
        "label": row["label"],
        "prompt_keys_in_window": prompt_keys,
        "completion_keys_in_window": int(
            row.get("completion_keys_in_window", 0)
        ),
        "valid_key_start": valid_key_start,
        "n_real_keys": n_real_keys,
    }

    output_row.update(
        dict(zip(feature_names, feature_vector))
    )
    output_rows.append(output_row)

    if (position + 1) % 20 == 0 or position + 1 == len(metadata):
        print(f"Processed {position + 1}/{len(metadata)}")


# ============================================================
# Save compact outputs
# ============================================================

X = np.stack(all_features).astype(np.float32)
X_prompt = np.stack(all_prompt_attention).astype(np.float32)

labels = np.asarray(
    [row["label"] for row in metadata],
    dtype="U32",
)

y = np.asarray(
    [1 if label == "correction" else 0 for label in labels],
    dtype=np.int8,
)

prompt_ids = np.asarray(
    [row["prompt_id"] for row in metadata],
    dtype="U64",
)

source_indices = np.asarray(
    [int(row["source_idx"]) for row in metadata],
    dtype=np.int32,
)

np.savez_compressed(
    FEATURE_NPZ,
    X=X,
    y=y,
    labels=labels,
    prompt_ids=prompt_ids,
    source_indices=source_indices,
    feature_names=np.asarray(feature_names, dtype="U32"),
)

np.savez_compressed(
    PROMPT_FEATURE_NPZ,
    X_prompt_attention=X_prompt,
    prompt_ids=prompt_ids,
    source_indices=source_indices,
    feature_names=np.asarray(
        prompt_feature_names,
        dtype="U40",
    ),
)

pd.DataFrame(output_rows).to_csv(
    FEATURE_CSV,
    index=False,
)


# ============================================================
# Validate and record checksums
# ============================================================

assert X.shape == (127, 1009), X.shape
assert X_prompt.shape == (127, 336), X_prompt.shape
assert int(y.sum()) == 22
assert np.isfinite(X).all()
assert np.isfinite(X_prompt).all()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


summary = {
    "condition": "reconstructed_full_context",
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "n_records": int(X.shape[0]),
    "n_features": int(X.shape[1]),
    "n_prompt_attention_features": int(X_prompt.shape[1]),
    "n_corrections": int(y.sum()),
    "n_non_corrective": int((y == 0).sum()),
    "feature_order": [
        "H1: 336",
        "H2: 336",
        "H3: 336",
        "H4: 1",
    ],
    "h3_definition": (
        "Captured causal-key attention entropy, renormalized "
        "within the captured structural key set and divided by log(K_q)."
    ),
    "files": {
        FEATURE_NPZ.name: sha256_file(FEATURE_NPZ),
        FEATURE_CSV.name: sha256_file(FEATURE_CSV),
        PROMPT_FEATURE_NPZ.name: sha256_file(
            PROMPT_FEATURE_NPZ
        ),
    },
}

with SUMMARY_JSON.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)


# ============================================================
# Create small download ZIP
# ============================================================

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

with ZipFile(
    OUTPUT_ZIP,
    mode="w",
    compression=ZIP_DEFLATED,
    compresslevel=9,
) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)


print("\nCompact Qwen bundle created successfully")
print(f"Feature matrix: {X.shape}")
print(f"Prompt-attention matrix: {X_prompt.shape}")
print(f"Corrections: {int(y.sum())}")
print(
    f"ZIP size: "
    f"{OUTPUT_ZIP.stat().st_size / 1024**2:.2f} MiB"
)
print(f"Download: {OUTPUT_ZIP}")

Metadata rows: 127
Processed 20/127
Processed 40/127
Processed 60/127
Processed 80/127
Processed 100/127
Processed 120/127
Processed 127/127

Compact Qwen bundle created successfully
Feature matrix: (127, 1009)
Prompt-attention matrix: (127, 336)
Corrections: 22
ZIP size: 1.15 MiB
Download: /kaggle/working/qwen_compact_analysis_bundle.zip
